# MLE — Train Arcade Game RL Agent on Colab GPU

Trains Dreamer v3 on any MAME arcade game using T4 GPU.

**Runtime**: Select `2025.07` (Python 3.11) under Runtime > Change runtime type.

**Upload your ROM zip** (e.g. `frogger.zip`) when prompted.

In [ ]:
# 1. Setup
!apt-get update -qq && apt-get install -y -qq mame xvfb > /dev/null 2>&1
!pip install -q sheeprl gymnasium pillow wandb opencv-python-headless 'numpy<2'

import subprocess, os
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1x1x24'])
os.environ['DISPLAY'] = ':99'
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['XDG_RUNTIME_DIR'] = '/tmp'
if '/usr/games' not in os.environ.get('PATH', ''):
    os.environ['PATH'] = '/usr/games:' + os.environ['PATH']

!rm -rf /content/mle && git clone https://github.com/lilwg/mle.git /content/mle
%cd /content/mle

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete!')

In [ ]:
# 2. Upload ROM
from google.colab import files
import os

uploaded = files.upload()
os.makedirs('roms', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'roms/{fname}')
rom_name = [f[:-4] for f in os.listdir('roms') if f.endswith('.zip')][0]
print(f'Game: {rom_name}')
!mame -rompath roms {rom_name} -verifyroms 2>&1

In [ ]:
# 3. Train Dreamer v3
GAME = rom_name
TIMESTEPS = 500_000

!cd /content/mle && \
    DISPLAY=:99 SDL_VIDEODRIVER=dummy SDL_AUDIODRIVER=dummy \
    MLE_ROMS_PATH=/content/mle/roms \
    python -u dreamer_train.py {GAME} --timesteps {TIMESTEPS}